# SOR

SPD methods solve f(x)=0.5*x.T@A@x-b.T@x. A small residual does not guarantee a small solution error. Stationary methods may diverge on an SPD matrix; inspect converged.

Run from the repository folder or any folder below it. Requires NumPy, SciPy and Numba.


In [ ]:
import time
import sys
from pathlib import Path
import numpy as np
from scipy.sparse import csr_matrix, issparse

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside Numerical_Linear_Algebra_Algorithms")
sys.path.insert(0, str(ROOT))
from common import loader, data_path, lu_solve, relative_residual
from Iterative_Methods.sor import sor


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


## 3×3


In [ ]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = sor(A, b)
result, seconds = timed(sor, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
assert relative_residual(A, x, b) < 1e-8


## SuiteSparse

Change MATRIX to an existing file. bcsstk09 is 1,083×1,083; bcsstk17 is 10,974×10,974; west1505 is nonsymmetric and requires pivoting. No 10^5 matrix exists in this repo. Keep CSR storage for larger tests.


In [ ]:
MATRIX = "bcsstk09"
MAT_PATH = None
A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=False)
print("Matrix:", MATRIX, "shape:", A.shape)
x_true = np.ones(A.shape[0])
b = A @ x_true


In [ ]:
result, seconds = timed(sor, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))


In [ ]:
RUN_10_4 = False

if RUN_10_4:
    MATRIX = "bcsstk17"
    MAT_PATH = None
    A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=False)
    print("Matrix:", MATRIX, "shape:", A.shape)
    x_true = np.ones(A.shape[0])
    b = A @ x_true
    result, seconds = timed(sor, A, b)
    print("Seconds:", seconds)
    x, info = result
    print(info)
    print("Relative residual:", relative_residual(A, x, b))
    print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
